In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

df = spark.read.table("ecommerce_analytics.bronze.sales")

product_schema = StructType([

  StructField('curr', StringType(), True),
  StructField('id', StringType(), True),
  StructField('name', StringType(), True),
  StructField('price', StringType(), True),
  StructField('qty', StringType(), True),
  StructField('unit', StringType(), True)
])

sales_df= df.withColumn("product",from_json("product",product_schema))\
    .select("customer_id","customer_name","product_name","order_date","product_category","product.*","total_price",)

sales_df.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.sales")





In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from delta.tables import DeltaTable

#####################################
############--SCD - 1--##############
#####################################

def scd_merge_table(spark, source_table, target_table, business_key):

    print("Checking if Silver Table exists")

    if not spark.catalog.tableExists(target_table):
        print("First Load: Creating Silver Table")

        source_table.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(target_table)

    else:
        print("Incremental Load: Performing SCD Type 1 Merge")

        delta_table = DeltaTable.forName(spark, target_table)

        merge_condition = " AND ".join(
            [f"target.{col} = source.{col}" for col in business_key]
        )

        (
            delta_table.alias("target")
            .merge(
                source_table.alias("source"),
                merge_condition
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    print("Merge Successfully")


####################################
########### SALES ##################
####################################

df = spark.read.table("ecommerce_analytics.bronze.sales")

product_schema = StructType([
    StructField("curr", StringType(), True),
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("price", StringType(), True),
    StructField("qty", StringType(), True),
    StructField("unit", StringType(), True)
])

sales_df = (
    df.withColumn(
        "product",
        from_json("product", product_schema)
    )
    .select(
        "customer_id",
        "customer_name",
        "product_name",
        "order_date",
        "product_category",
        "product.*",
        "total_price"
    )
    .dropDuplicates(["customer_id", "id", "order_date"])
)

####################################
########### WRITE ##################
####################################

scd_merge_table(
    spark,
    sales_df,
    "ecommerce_analytics.silver.sales",
    ["customer_id", "id", "order_date"]
)

print("Successfully wrote sales table")

In [0]:
display(sales_df.limit(10))